# Repository

In [1]:
import os          
import pathlib     
import sys         

here = pathlib.Path.cwd()      

ROOT = here.parents[2] if here.name == "0911F" else here
os.chdir(ROOT)                 

SANDBOX = ROOT / "sandbox" / "w2" / "0911F"

BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

In [2]:

from datetime import date
from app.db.session import get_sessionmaker, get_engine
from app.models import Department, Document, DocumentVersion
from app.db.init_db import init_db

# 연습용 DB 경로
연습DB = SANDBOX / "repo_practice.db"
연습DB.unlink(missing_ok=True)               # 파일이 없어도 오류 아님ㅇㅇ처리                 

# 연습용 DB 환경 설정
연습엔진 = get_engine(f"sqlite:///{연습DB}")
init_db(연습엔진)                   
연습세션 = get_sessionmaker(연습엔진)  

# SQL 연습
with 연습세션() as s:
    # 부서 저장
    s.add_all([
        Department(id="HRGA", name="인사총무"),
        Department(id="PU", name="구매팀"),
        Department(id="SE", name="보안팀"),
    ])
    s.flush()      # 쿼리문에서 나가서-> DB에 저장됨

    # 문서 서장
    s.add_all([
        Document(id="DOC-HR-014", title="국내출장 여비 규정",
                 dept_id="HRGA", security_level="일반"),
        Document(id="DOC-PU-007", title="구매·계약 규정",
                 dept_id="PU", security_level="대외비"),
        Document(id="DOC-SE-003", title="정보보안 지침",
                 dept_id="SE", security_level="대외비"),
    ])
    s.flush()       # 쿼리문에서 나가서-> DB에 저장됨

    # 버전별 문서 저장 - 문서마다 각 버전별로 저장될 수 있음
    s.add_all([
        DocumentVersion(doc_id="DOC-HR-014", version="v2.0", status="현행",
                        effective_from=date(2025, 7, 1), expires_at=None,
                        file_path="uploads/DOC-HR-014_v2.0.docx", file_format="docx"),
        DocumentVersion(doc_id="DOC-PU-007", version="v4.0", status="현행",
                        effective_from=date(2026, 3, 1), expires_at=None,
                        file_path="uploads/DOC-PU-007_v4.0.pdf", file_format="pdf"),
        DocumentVersion(doc_id="DOC-SE-003", version="v2.2", status="현행",
                        effective_from=date(2025, 10, 1), expires_at=None,
                        file_path="uploads/DOC-SE-003_v2.2.pdf", file_format="pdf"),
    ])
    s.commit()      # DB에 영구 저장

print("연습 DB   :", 연습DB.relative_to(ROOT))
print("문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)")


연습 DB   : sandbox\w2\0911F\repo_practice.db
문서 3건 · 버전 3건 준비 완료 (대외비 2건 포함)


In [3]:
from sqlalchemy import select

stmt = select(Document).where(Document.dept_id == "HRGA")

print(stmt)                                  
print()
print("바인딩된 값 :", stmt.compile().params)

SELECT documents.id, documents.title, documents.dept_id, documents.security_level, documents.owner_id, documents.created_at, documents.updated_at 
FROM documents 
WHERE documents.dept_id = :dept_id_1

바인딩된 값 : {'dept_id_1': 'HRGA'}


In [4]:
from app.repositories import document_repo

jogun_list=[
    ('dept_id=SE',{'dept_id':'SE'}),
    ('security_level=대외비',{'security_level':'대외비'}),
    ('q=출장',{'q':'출장'}),
    ('dept_id=SE + status=현행',{'dept_id':'SE', 'status':'현행'}),
]

with 연습세션() as session:
    for name, jogun in jogun_list:
        results=document_repo.list_documents(session, **jogun)
        view='.'.join(f"{v.doc_id}{v.version}" for v, d in results)
        print(f"{name:}->{len(results)}, {view}")

'''
with 연습세션() as session: # DB 접속은 session
    print('조건 없음')
    result=document_repo.list_documents(session)
    print(len(result)) # 몇 개
'''
'''
for row in result:
print(row)
'''
'''
show=','.join(f"{v.doc_id}{v.version}" for v, d in result)
print(f"")
'''

dept_id=SE->1, DOC-SE-003v2.2
security_level=대외비->2, DOC-PU-007v4.0.DOC-SE-003v2.2
q=출장->1, DOC-HR-014v2.0
dept_id=SE + status=현행->1, DOC-SE-003v2.2


'\nshow=\',\'.join(f"{v.doc_id}{v.version}" for v, d in result)\nprint(f"")\n'

In [5]:
from app.db.seed import count_rows, seed_all
from app.db.session import session_scope

init_db()
print(seed_all())

{'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}


In [6]:
from app.services import document_service

목록=document_service.list_documents(limit=20)
print('count the list:',len(목록),'ae')
print('a document:',목록[0]['doc_id'],목록[0]['version'],목록[0]['title'])

문서=document_service.get_document(doc_id='DOC-HR-014')
print('문서:',문서['version'],문서['status'])

count the list: 8 ae
a document: DOC-HR-002 v3.1 복무 규정
문서: v2.0 현행


## Schema

In [7]:
from app.services import document_service

hangs=document_service.list_documents()
print('서비스 반환한 문서 개수:', len(hangs))
print('문서 한 개의 값 이름들:', sorted(hangs[0]))

서비스 반환한 문서 개수: 8
문서 한 개의 값 이름들: ['dept', 'doc_id', 'effective_from', 'expires_at', 'file_format', 'index_progress', 'index_status', 'security_level', 'status', 'title', 'version']
